# 06 — Evaluation, SHAP, and driver analysis

**Workstream**: Eval + SHAP  ·  **Owner**: Bella (backup: Deepak)  ·  **Last touched**: 2026-06-15

**What this notebook does**

1. **Group-performance audit.** Does the served model perform comparably across
   `static_facility_type` and full `static_zip`? Basic in-scope fairness check
   (full disparate-impact audit is Phase 2). Run on the **right-truncation
   filtered** test split, matching the served eval basis.
2. **Per-restaurant SHAP top drivers.** For every restaurant, compute the 3-5
   features doing most of the work behind its predicted score, with plain-English
   labels (`src/foodsafety/explain/shap_drivers.py`). These feed the detail page.

> **Scoring / contract artifacts are NOT written here.** The served
> `scores.parquet` + `scores.json` are produced by the single writer,
> `scripts/retrain_baseline_sigmoid.py` (sigmoid-calibrated). This notebook is
> analysis only — see § 6.

**Production estimator**: calibrated baseline logistic regression (sigmoid,
served via the script). Feature set is whatever `baseline.py::ALL_FEATURES`
currently pins. Per CLAUDE.md's gate, XGBoost did not clear baseline on PR-AUC
AND precision@10%, so baseline ships. SHAP for logreg has a closed form
(coef × scaled feature value) so we compute it directly without the `shap`
package.

## 1. Setup

In [ ]:
import sys
import json
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent
if str(_PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from foodsafety.config import MODELS_DIR, PROCESSED_DIR
from foodsafety.utils.time import temporal_split
from foodsafety.models.baseline import ALL_FEATURES, LABEL_COL
from foodsafety.models.evaluate import evaluate, decile_lift_table, group_performance_audit
from foodsafety.features.license_features import normalize_facility_type
from foodsafety.explain.shap_drivers import linear_contributions, top_drivers_for_row, FEATURE_LABELS
from foodsafety.serve.predict_batch import build_scores_table, write_scores_json, score_to_tier

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 2. Load the production model + features

Pick the most recent `baseline_*.joblib` from `data/models/`. The same
stamp file's `metadata.json` records the cutoffs we'll re-use.

In [ ]:
model_files = sorted(MODELS_DIR.glob('baseline_[0-9]*.joblib'))
if not model_files:
    raise SystemExit('No baseline model found. Run notebook 04 first.')

model_path = model_files[-1]
meta_path = model_path.with_name(model_path.stem + '_metadata.json')

model = joblib.load(model_path)
meta = json.loads(meta_path.read_text())

print(f'model:    {model_path.name}')
print(f'metadata: train_end={meta["split"]["train_end"]}, val_end={meta["split"]["val_end"]}')
print(f'features: {len(meta["features"]["all"])}')

features = pd.read_parquet(PROCESSED_DIR / 'features.parquet')
features['inspection_date'] = pd.to_datetime(features['inspection_date'])
for c in features.columns:
    if c.startswith('flag_kw_'):
        features[c] = features[c].astype('int8')
print(f'features: {len(features):,} rows × {features.shape[1]} cols')

## 3. Group-performance audit

Does the served model rank comparably **across kinds of establishment and across the
city**? We compute per-group PR-AUC, precision@10%, and recall@10% (coverage) for each
`facility_type` and full 5-digit `static_zip`, via the reusable
`evaluate.group_performance_audit` — the same check any new experiment should run.
`facility_type` is normalized with `normalize_facility_type` so vulnerable-population
families (daycare, school, …) aren't fragmented across spellings.

  - Rule (CLAUDE.md): no group below **50% of overall PR-AUC**; groups with n < 50 are
    skipped (noisy). We do NOT use `static_zip3` — in Chicago it's ~one bucket ("606").
  - **Caveat:** PR-AUC is mechanically low at low prevalence, so a below-floor
    rare-event group is usually a base-rate artifact, not bias — read it alongside
    recall@10% (coverage). The full disparate-impact audit (demographic join) is Phase 2.
  - Right-truncation filtered to match the served eval basis. The recorded verdict +
    interpretation live in `docs/fairness_audit.md`.

In [ ]:
# RT-FILTER to match the served eval basis. Right-truncated rows (their 180-day
# forward window extends past the dataset max date) have UNDER-COUNTED labels, so
# the served script (retrain_baseline_sigmoid.py) drops them before splitting.
features_eval = (
    features.loc[~features['right_truncated']].reset_index(drop=True)
    if 'right_truncated' in features.columns else features
)

split = temporal_split(
    features_eval,
    train_end=meta['split']['train_end'],
    val_end=meta['split']['val_end'],
)
test = split.test.copy()
test['risk_score'] = model.predict_proba(test[ALL_FEATURES])[:, 1]
test['y'] = test[LABEL_COL].astype(int)

# Normalize facility_type so vulnerable-pop families aren't fragmented across spellings
# (audit/UI only; static_facility_type is dropped from the model — DR 0004).
test['facility_group'] = test['static_facility_type'].astype('string').map(normalize_facility_type)

print(f'test rows (RT-filtered): {len(test):,}')
fac_audit = group_performance_audit(test['y'], test['risk_score'], test['facility_group'])
print(
    f"overall PR-AUC={fac_audit.attrs['overall_pr_auc']}  "
    f"floor ({fac_audit.attrs['floor_frac']:.0%} of overall)={fac_audit.attrs['pr_auc_floor']}"
)
print('\n--- by facility (normalized) ---')
print(fac_audit.to_string(index=False))

In [ ]:
# Geography audit on FULL static_zip (static_zip3 is ~one Chicago bucket, "606").
# static_zip stays in the parquet for this audit though it's dropped from the model
# (DR 0004).
zip_audit = group_performance_audit(test['y'], test['risk_score'], test['static_zip'])
print('--- by ZIP (5-digit, top 15 by n) ---')
print(zip_audit.head(15).to_string(index=False))

# Combined below-floor report across both axes (facility + ZIP).
below = pd.concat(
    [
        fac_audit[fac_audit['below_floor']].assign(axis='facility'),
        zip_audit[zip_audit['below_floor']].assign(axis='zip'),
    ],
    ignore_index=True,
)
floor = fac_audit.attrs['pr_auc_floor']
print(f"\nGroups below the 50%-of-overall PR-AUC floor ({floor}):")
if len(below):
    print(below[['axis', 'group', 'n', 'positive_rate', 'pr_auc', 'recall_at_k']].to_string(index=False))
    print('\nNote: PR-AUC is mechanically low at low prevalence — read these alongside')
    print('recall@10% (coverage); sub-~50-positive groups are noise (DR 0005). See')
    print('docs/fairness_audit.md for the recorded verdict + interpretation.')
else:
    print('  none — all audited groups clear the floor.')

## 4. Global feature impact (mean |contribution|)

Per-feature mean of `|log-odds contribution|` on the test set. This is the
global view that goes in the model card. The detail-page driver bars use
per-row contributions (computed in § 5).

In [ ]:
contrib_test = linear_contributions(model, test[ALL_FEATURES], original_features=ALL_FEATURES)

global_impact = contrib_test.abs().mean().sort_values(ascending=False)
print('Top 15 features by mean |log-odds contribution|:')
print(global_impact.head(15).round(4).to_string())

ax = global_impact.head(15)[::-1].plot.barh(color='#15110D', figsize=(8, 6))
ax.set_xlabel('Mean |log-odds contribution|')
ax.set_title('Production model — global feature impact (test set)')
plt.tight_layout(); plt.show()

### 4a. Leave-one-out feature ablation (diagnostic)

Which features actually carry the model? Drop each `ALL_FEATURES` column one at a
time, retrain XGBoost on the same chronological split, and measure the change in
honest-test PR-AUC / precision@10%. (XGBoost, not the served LogReg — leave-one-out
needs a retrain per feature, and XGB handles NaN / categoricals natively. We re-split
from the **full** `features` so the numbers match the row logged in
`docs/experiments.md`, not the RT-filtered audit split in § 3.)

**Read it with three caveats:**
1. **Single split.** A small *positive* ΔPR-AUC from *removing* a feature is noise,
   not a mandate to drop it — any cut must clear expanding-window CV first, the same
   bar additions face.
2. **Leave-one-out understates correlated features.** `was_fail` and
   `n_priority_this_inspection` mask each other (dropping one, the other compensates),
   so their *joint* importance is larger than either row shows.
3. **Small-group fairness still applies.** A feature that looks like dead weight here
   may matter for a vulnerable-population subgroup — cross-check § 3 before cutting.

In [ ]:
# Leave-one-out ablation. Retrains XGBoost per dropped feature, so it builds its
# own models from the split (the loaded `model` is the served LogReg, which can't
# be ablated in place). Re-split from the FULL `features` (honest-test basis,
# n~13,812) so the deltas line up with the row in docs/experiments.md — NOT the
# RT-filtered `split` used for the served fairness audit above.
from foodsafety.models.baseline import NUMERIC_FEATURES, CATEGORICAL_FEATURES, BOOLEAN_FEATURES
from foodsafety.models.xgb import build_xgb_estimator, compute_scale_pos_weight

_abl = temporal_split(features, train_end=meta['split']['train_end'], val_end=meta['split']['val_end'])
_y = {k: getattr(_abl, k)[LABEL_COL].astype(int) for k in ('train', 'val', 'test')}
_spw = compute_scale_pos_weight(_y['train'])


def _prep_xgb(frame, feats, cats=None):
    """Cast a feature subset to XGBoost dtypes; reuse train's categories on val/test."""
    out = frame[feats].copy()
    newcats = {}
    for c in feats:
        if c in CATEGORICAL_FEATURES:
            out[c] = (pd.Categorical(out[c], categories=cats[c].categories)
                      if cats and c in cats else out[c].astype('category'))
            newcats[c] = out[c].dtype
        elif c in BOOLEAN_FEATURES:
            out[c] = out[c].astype('int8')
        else:
            out[c] = out[c].astype('float32')
    return out, newcats


def _fit_eval(feats):
    Xtr, cats = _prep_xgb(_abl.train, feats)
    Xval, _ = _prep_xgb(_abl.val, feats, cats)
    Xte, _ = _prep_xgb(_abl.test, feats, cats)
    m = build_xgb_estimator(scale_pos_weight=_spw)
    m.fit(Xtr, _y['train'], eval_set=[(Xval, _y['val'])], verbose=False)
    r = evaluate(_y['test'], m.predict_proba(Xte)[:, 1])
    return r.pr_auc, r.precision_at_10pct


_base_pr, _base_p10 = _fit_eval(ALL_FEATURES)
print(f'full XGB ({len(ALL_FEATURES)} feats): '
      f'PR-AUC={_base_pr:.4f}  P@10={_base_p10:.4f}  (test n={len(_y["test"]):,})')

loo = pd.DataFrame(
    [
        dict(zip(('feature', 'd_pr_auc', 'd_p10'),
                 (f, *np.subtract(_fit_eval([x for x in ALL_FEATURES if x != f]),
                                  (_base_pr, _base_p10)))))
        for f in ALL_FEATURES
    ]
).sort_values('d_pr_auc').reset_index(drop=True)  # most negative = removing it hurts most

print('\nLeave-one-out Δ (removing the feature); negative ΔPR-AUC = the feature matters:')
print(loo.round(4).to_string(index=False))

ax = (loo.set_index('feature')['d_pr_auc'].head(12)[::-1]
      .plot.barh(color='#15110D', figsize=(8, 5)))
ax.set_xlabel('Δ PR-AUC when feature is removed (negative = feature matters)')
ax.set_title('Leave-one-out feature ablation (XGBoost, honest test)')
plt.tight_layout()
plt.show()

## 5. Score every restaurant + build the contract artifact

`build_scores_table` produces one row per `license_id` anchored on the
restaurant's most recent inspection. It computes risk score, risk tier,
top-4 drivers (plain-English labels), and 90-day trend slope.

Output schema matches `docs/interface_contracts.md` § 3.

Wall-clock: ~30-60 s. Most of the time is the per-row SHAP attribution +
trend OLS fit; could be vectorised further if it grows.

In [ ]:
%%time
scores = build_scores_table(
    model=model,
    features=features,
    feature_columns=ALL_FEATURES,
    n_drivers=4,
)
print(f'\nscores: {len(scores):,} rows × {scores.shape[1]} cols')

In [ ]:
# Quick eyeball — top 10 by risk_score
preview = scores.sort_values('risk_score', ascending=False).head(10)
preview[['license_id', 'dba_name', 'address', 'risk_score', 'risk_tier', 'trend_slope_90d']].to_string()

In [ ]:
# Tier distribution
print('Tier distribution across all restaurants:')
print(scores['risk_tier'].value_counts().to_string())
print()

# Score distribution
print(scores['risk_score'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(3).to_string())

ax = scores['risk_score'].plot.hist(bins=40, color='#15110D', alpha=0.85,
                                     title='Risk-score distribution across restaurants')
ax.set_xlabel('Risk score'); plt.tight_layout(); plt.show()

### 5a. Sample driver lists

Spot-check three restaurants — one from each of High, Elevated, Low — and
print their top drivers. The labels are what end up on the detail page.

In [ ]:
for tier in ['High', 'Elevated', 'Low']:
    subset = scores[scores['risk_tier'] == tier]
    if subset.empty:
        continue
    sample = subset.iloc[0]
    print(f'--- {tier}: {sample["dba_name"]} ({sample["risk_score"]:.2f}) ---')
    for d in sample['top_drivers']:
        sign = '+' if d['shap'] > 0 else '−'
        print(f'  · {d["label"]:<55} ({sign}{abs(d["shap"]):.3f})')
    print()

## 6. Scoring — produced by the served script, NOT this notebook

The served `data/predictions/scores.parquet` and `app/public/data/scores.json`
are written by **`scripts/retrain_baseline_sigmoid.py`** — the *single writer*,
using the **sigmoid-calibrated** served model (decision records 0001 / 0002).

This notebook computes `scores` in-memory above only for SHAP / driver
inspection (§ 5) and **does not write the served artifacts**. Writing them here
from the *isotonic* baseline was a dual-writer that reintroduced the isotonic
score-tie problem the sigmoid switch fixed — removed per the methodology review.

In [ ]:
# The served scores.parquet is written by scripts/retrain_baseline_sigmoid.py
# (single writer, sigmoid). This notebook does NOT write it — we only inspect the
# in-memory `scores` table for SHAP / driver analysis.
print('tier distribution (in-memory `scores`, NOT written):')
print(scores['risk_tier'].value_counts().to_string())
print()
print(scores['risk_score'].describe(percentiles=[0.5, 0.9, 0.99]).round(3).to_string())

In [ ]:
# The served app/public/data/scores.json is produced by the sigmoid script,
# NOT here. (This cell previously wrote it from the ISOTONIC model — a dual-writer
# that reintroduced UI score-ties; removed per the methodology review / DR 0001.)
print('scores.json is produced by scripts/retrain_baseline_sigmoid.py (single writer, sigmoid).')

## 7. Hand-off

**Outputs**:
  - `data/predictions/scores.parquet` — Python pipeline artifact (cross-team contract)
  - `app/public/data/scores.json` — Next.js input. The web app **auto-drops
    the demo banner** when this file replaces the `scores_mock.json` fallback.

**Verification**: `cd app && pnpm dev` (or `npm run dev`) and visit
`http://localhost:3000` — the yellow demo banner should be gone, and the
real restaurant scores should appear.

**The walking-skeleton transition is complete.** Phase 1 mocked everything;
Phases 2-6 swapped each component for real implementations. The architecture
is the same; only the data changed.